# Detecting Persistent Memory Poisoning on `ap_burst2k`

Reproduction notebook for the amended results. Moses W. Ayandau, PG3996225, KNUST MSc Cyber
Security and Digital Forensics.

**This notebook supersedes the AUROC 1.0000 result.** That figure was measured on
`data/raw/llm_generated`, where a logistic regression given a single record's 384-dimensional
embedding -- no graph, no edges, no temporal order -- also scores AUROC 1.0000. The label was
recoverable from one record's text, so no graph model was being tested. `ap_burst2k` is the
replacement benchmark, and on it the same probe scores 0.5300.

**Headline figure: sequence-level AUROC 0.8122, AP 0.8185.**

### What runs here

| Section | What it does | Roughly how long |
|---|---|---|
| 1 | Environment and reproducibility | seconds |
| 2 | Validate the benchmark before training on it | ~5 min |
| 3 | Configuration | seconds |
| 4 | Train the detector | ~5 h (or load a checkpoint) |
| 5 | Sequence-level results -- the headline | ~2 min |
| 6 | All six baselines | ~6 h to retrain, or load |
| 7 | Calibrated detection latency | ~5 min |
| 8 | The thesis table | seconds |

Every long step has a `RUN_*` switch. With all of them `False` the notebook loads existing
artefacts and completes in a few minutes; that is the mode to use when checking the figures.
Set one `True` to recompute that piece from scratch.

### A note on what "reproducible" means here

Re-running evaluation on the *same* checkpoint and data in a fresh process moves graph AUROC
by about +/-0.007 and node AP by about +/-0.014. Training from scratch under a different seed
moved the sequence-level AUROC from 0.8122 (seed 42) to 0.8140 (seed 1), a gap of 0.0018.
**Treat differences below about 0.02 as noise**, and do not report a difference that small as
a finding.

## 1. Environment and reproducibility

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch
import yaml
from sklearn.metrics import roc_auc_score, average_precision_score

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)   # so the module CLIs below resolve their relative paths

from src.models.train import set_seed

SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"repo root : {REPO_ROOT}")
print(f"python    : {sys.version.split()[0]}")
print(f"torch     : {torch.__version__}")
print(f"device    : {device}")
if device.type == "cuda":
    print(f"gpu       : {torch.cuda.get_device_name(0)}")
try:
    sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
    branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip()
    print(f"commit    : {sha} on {branch}")
except Exception as exc:
    print(f"commit    : unavailable ({exc})")
print(f"seed      : {SEED}")

## 2. The benchmark, and why it is trustworthy

`ap_burst2k` is built from AgentPoison's published StrategyQA knowledge base (Chen et al.,
2024). **Every record, benign or poisoned, is a genuine unmodified entry from that base.** The
same entry appears as benign in one sequence and poisoned in another, so there is no text
difference for a content classifier to find. What makes a sequence poisoned is *placement*:
the attacker floods one narrow embedding neighbourhood in a burst.

Each clean sequence receives a **matched benign cluster** of the same size, drawn from an
equally narrow neighbourhood, but arriving gradually across 120 interactions instead of in a
burst. The final snapshot therefore carries the same density either way, and **burstiness is
the only discriminating signal**.

This matters because the previous benchmark failed for the opposite reason: its poisoned
records were *generated* from a different instruction, so poisoned and benign text were two
distributions and a classifier separated them from a single record.

In [ ]:
DATA_DIR = REPO_ROOT / "data" / "raw" / "ap_burst2k"
assert DATA_DIR.exists(), f"missing {DATA_DIR} -- regenerate it with the command below"

RUN_REGENERATE = False   # True rebuilds the dataset from scratch (~30 min)

GENERATE_CMD = [
    sys.executable, "-m", "src.data_generation.agentpoison_sequences",
    "--output", "data/raw/ap_burst2k", "--num-sequences", "2000",
    "--matched-control", "1.0", "--benign-spread", "120",
    "--attack-spread-mult", "1.0", "--seed", "42",
]
if RUN_REGENERATE:
    print(" ".join(GENERATE_CMD))
    subprocess.run(GENERATE_CMD, check=True)

meta = {m["sequence_id"]: m for m in json.load(open(DATA_DIR / "metadata.json"))}
splits = json.load(open(DATA_DIR / "splits.json"))
n_pois = sum(1 for m in meta.values() if m["label"] == "poisoned")

print(f"sequences : {len(meta)}  ({n_pois} poisoned, {len(meta) - n_pois} clean)")
for name in ("train", "val", "test"):
    ids = splits[name]
    p = sum(1 for s in ids if meta[s]["label"] == "poisoned")
    print(f"  {name:<6}: {len(ids):>5} sequences, {p:>4} poisoned")
print("\nexpected: 2000 total (800 poisoned) -- 1200 train / 300 val / 500 test,")
print("          test split 500 sequences with 200 poisoned")

### 2.1 The validation gate

Four ways a memory-poisoning benchmark can be solvable without doing any of the work it claims
to test. Each has been hit in this project, and each is measured **before** a GPU is used.

1. **Content leak** -- a logistic regression on one record's embedding recovers the label.
   This is what invalidated `llm_generated` at AUROC 1.0000.
2. **Position leak** -- window labels are presence-based, and `n_nodes(snapshot i) =
   (i + 1) * snapshot_interval`, so node count *is* window position under another name.
3. **Final-state leak** -- the poisoned final graph is simply denser, so no temporal
   modelling is needed. This is what the matched benign cluster exists to remove.
4. **No signal at all** -- having removed the leaks, nothing is left to learn. A benchmark can
   be unsolvable rather than hard, and the two look identical from a training curve.

In [ ]:
RUN_GATE = False   # True re-measures the four controls (~5 min, CPU)

GATE_CMD = [sys.executable, "-m", "src.experiments.benchmark_gate",
            "--data", "data/raw/ap_burst2k", "--split", "all", "--max-sequences", "800"]
if RUN_GATE:
    print(" ".join(GATE_CMD))
    subprocess.run(GATE_CMD, check=True)

gate_path = DATA_DIR / "benchmark_gate.json"
if gate_path.exists():
    gate = json.load(open(gate_path))
    print(f"position-only (window level) : {gate['position_only_window_auroc']:.4f}")
    print(f"verdict                      : {gate['verdict']}")
else:
    print(f"(not found: {gate_path} -- set RUN_GATE = True)")

print("""
Recorded in the thesis, all measured:

  control                          llm_generated   ap_burst2k    verdict
  per-record content probe            1.0000         0.5300      chance
  pooled content probe                0.7729         0.5166      chance
  position alone (window level)       0.5143         0.6086      reported explicitly
  final-state features (sequence)        --          0.4650      chance, CI [0.415, 0.519]
  node count (sequence level)            --          0.5000      inert by construction
  temporal features (sequence)           --          0.7934      signal present

Position is irrelevant at sequence level because every sequence has identical length and node
count. n_nodes scoring exactly 0.5000 is that fact stated as a measurement. The window-level
0.6086 is a real floor and must never be compared against a sequence-level number.
""")

## 3. Configuration

In [ ]:
# Read from configs/default.yaml rather than restated here. A second copy of the
# hyperparameters is a second thing to keep in sync, and the training CLI reads the file --
# restating it is how a notebook silently trains a different model from the one reported.
CONFIG_PATH = REPO_ROOT / "configs" / "default.yaml"
CONFIG = yaml.safe_load(open(CONFIG_PATH))

EXP_DIR = REPO_ROOT / "experiments" / "run_burst2k_base"
EXP_DIR.mkdir(parents=True, exist_ok=True)

READOUT = "attention"          # "attention" | "scale" | "gated"
STRUCTURAL_FEATURES = True     # must match the checkpoint being loaded
EPOCHS_SUPERVISED = 12         # the config default of 10 is too few on this benchmark

print(f"config: {CONFIG_PATH}")
for section in ("simulation", "training", "model"):
    print(f"\n[{section}]")
    for k, v in CONFIG[section].items():
        print(f"  {k}: {v}")

## 4. Train the detector

Phase 1 (contrastive pre-training) is skipped in every current run, matching `--skip-phase1`
on the command line. Phase 2 is the supervised stage.

`structural_features=True` is **not optional** if you want to reproduce these numbers: it adds
four per-node columns (degree under two scalings, and the mean and max weight of a node's
incident edges) on top of the 384-d embedding and 3 metadata columns, and the model's input
width depends on it. Loading a checkpoint into a differently-shaped model produces something
that looks like a number and means nothing.

Checkpoint selection is on validation **AUROC**, not validation loss. Selecting on loss picks
the wrong model whenever the two diverge, which they do on any benchmark hard enough for the
model to start overfitting.

In [ ]:
RUN_TRAINING = False   # True retrains from scratch: 12 epochs, roughly 5 hours on a 4GB card

from src.models.train import Trainer

if RUN_TRAINING:
    cfg = json.loads(json.dumps(CONFIG))
    cfg["training"]["epochs_supervised"] = EPOCHS_SUPERVISED
    set_seed(SEED)
    trainer = Trainer(
        cfg, str(DATA_DIR), device=device,
        noise_augment_prob=0.5, noise_sigma_max=0.15,
        structural_features=STRUCTURAL_FEATURES,
        select_by="auroc",
        readout=READOUT,
    )
    trainer.train_phase2_supervised(str(EXP_DIR))
    print(f"\nCheckpoints in {EXP_DIR}")
else:
    print(f"RUN_TRAINING = False -- using the existing checkpoint in {EXP_DIR}")
    print("\nEquivalent command line:")
    print(f"  python -m src.models.train --data data/raw/ap_burst2k \\")
    print(f"    --output experiments/run_burst2k_base --structural-features \\")
    print(f"    --readout {READOUT} --skip-phase1 --select-by auroc \\")
    print(f"    --epochs-supervised {EPOCHS_SUPERVISED} --seed {SEED}")

hist_path = EXP_DIR / "training_history.json"
if hist_path.exists():
    sel = json.load(open(hist_path))["selection"]
    print(f"\nselected checkpoint: epoch {sel['best_epoch']} "
          f"(val {sel['criterion']}={sel['best_score']:.4f})")
    print("expected: epoch 9, val auroc=0.8979")

## 5. Sequence-level results -- the headline

**This is the section that produces the figure for the thesis.**

The question is "is this agent's memory poisoned?", which is a *per-sequence* question: one
observation per sequence, 500 in the test split, 200 poisoned. The detector scores windows, so
window scores are aggregated to one score per sequence.

Three aggregators are reported rather than one. Choosing the best after seeing the test
results would be selection on the test set.

The cells below are drivers over `src/experiments/sequence_eval.py`: they import its feature
builder, its baseline fitting, its bootstrap and its paired test, so the notebook and the
command line cannot disagree.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from src.experiments.sequence_eval import (
    FINAL_FEATS, TEMPORAL_FEATS, sequence_features, to_matrix, boot_ci, paired_boot,
)

N_BOOT = 2000   # the CLI default; the reported intervals use this value

cache_path = DATA_DIR / "sequence_features.json"
if cache_path.exists():
    cached = json.load(open(cache_path))
    print(f"loaded cached sequence features from {cache_path}")
else:
    print("computing sequence features (a few minutes; cached afterwards)...")
    cached = {name: sequence_features(DATA_DIR, splits[name],
                                      CONFIG["simulation"]["semantic_edge_threshold"],
                                      CONFIG["simulation"]["embedding_dim"])
              for name in ("train", "val", "test")}
    json.dump(cached, open(cache_path, "w"))

y_train = np.array([int(meta[r["sequence_id"]]["label"] == "poisoned") for r in cached["train"]])
y_test = np.array([int(meta[r["sequence_id"]]["label"] == "poisoned") for r in cached["test"]])
test_order = [r["sequence_id"] for r in cached["test"]]
print(f"{len(y_train)} train sequences ({y_train.sum()} poisoned), "
      f"{len(y_test)} test sequences ({y_test.sum()} poisoned)")

In [ ]:
# The three logistic-regression baselines.
#
# "all features (13)" is an ORACLE: its features were hand-built from knowledge of how the
# attack is generated. Beating it is a strong claim; tying it is not a failure.
seq_scores = {}

for name, feats in (("final-state features (6)", FINAL_FEATS),
                    ("temporal features (7)", TEMPORAL_FEATS),
                    ("all features (13)", FINAL_FEATS + TEMPORAL_FEATS)):
    Xtr, Xte = to_matrix(cached["train"], feats), to_matrix(cached["test"], feats)
    scaler = StandardScaler().fit(Xtr)
    clf = LogisticRegression(max_iter=5000, class_weight="balanced").fit(
        scaler.transform(Xtr), y_train)
    seq_scores[name] = clf.decision_function(scaler.transform(Xte))

print(f"{'model':<32} {'AUROC':>7} {'AP':>7}   {'95% CI (AUROC)':>18}   expected")
print("-" * 92)
EXPECT_LR = {"final-state features (6)": (0.4650, 0.3872),
             "temporal features (7)": (0.7934, 0.7108),
             "all features (13)": (0.8030, 0.7203)}
for name, s in seq_scores.items():
    lo, hi = boot_ci(y_test, s, N_BOOT)
    e = EXPECT_LR[name]
    print(f"{name:<32} {roc_auc_score(y_test, s):>7.4f} "
          f"{average_precision_score(y_test, s):>7.4f}   [{lo:.4f}, {hi:.4f}]   "
          f"{e[0]:.4f} / {e[1]:.4f}")

In [ ]:
# The detector, aggregated from window scores to one score per sequence.
from torch.utils.data import DataLoader
from src.data_generation.dataset import MemoryTrajectoryDataset, collate_trajectories
from src.models.tgn_detector import TGNDetector

RESCORE_DETECTOR = False   # True re-runs the model over the 5,500 test windows (~3 min)

SEQ_CACHE = EXP_DIR / "sequence_scores_test.npz"
if SEQ_CACHE.exists() and not RESCORE_DETECTOR:
    z = np.load(SEQ_CACHE, allow_pickle=True)
    assert (z["y"] == y_test).all(), "cached labels disagree with the freshly computed ones"
    for key in z.files:
        if key.startswith("T-GNN"):
            seq_scores[key.replace("_", " ")] = z[key]
    print(f"loaded detector sequence scores from {SEQ_CACHE}")
else:
    cfg = json.loads(json.dumps(CONFIG))
    cfg["model"]["node_meta_dim"] = 3 + 4
    cfg["model"]["readout"] = READOUT
    model = TGNDetector(cfg).to(device)
    model.load_state_dict(torch.load(EXP_DIR / "phase2_best.pt",
                                     map_location=device, weights_only=True))
    model.eval()

    ds = MemoryTrajectoryDataset(str(DATA_DIR), split="test",
                                 temporal_window=CONFIG["training"]["temporal_window"],
                                 embedding_dim=CONFIG["simulation"]["embedding_dim"],
                                 structural_features=STRUCTURAL_FEATURES)
    loader = DataLoader(ds, batch_size=CONFIG["training"]["batch_size"], shuffle=False,
                        collate_fn=collate_trajectories)
    model.reset_memory()
    win = []
    with torch.no_grad():
        for batch in loader:
            graphs = [g.to(device) for g in batch["graphs"]]
            s, _ = model(graphs)
            win.extend(s.cpu().numpy().tolist())
    win = np.array(win, float)
    wseq = np.array([w[0] for w in ds.windows])
    for agg, fn in (("max", np.max), ("mean", np.mean), ("last", lambda v: v[-1])):
        seq_scores[f"T-GNN baseline (window {agg})"] = np.array(
            [fn(win[wseq == sid]) if (wseq == sid).any() else 0.0 for sid in test_order], float)
    print(f"scored {len(win)} test windows")

print()
print(f"{'model':<32} {'AUROC':>7} {'AP':>7}   {'95% CI (AUROC)':>18}   expected")
print("-" * 92)
EXPECT_TGNN = {"T-GNN baseline (window max)": (0.8122, 0.8185),
               "T-GNN baseline (window mean)": (0.8120, 0.8003),
               "T-GNN baseline (window last)": (0.7545, 0.7545)}
for name, s in seq_scores.items():
    if not name.startswith("T-GNN"):
        continue
    lo, hi = boot_ci(y_test, s, N_BOOT)
    e = EXPECT_TGNN.get(name, (float("nan"), float("nan")))
    print(f"{name:<32} {roc_auc_score(y_test, s):>7.4f} "
          f"{average_precision_score(y_test, s):>7.4f}   [{lo:.4f}, {hi:.4f}]   "
          f"{e[0]:.4f} / {e[1]:.4f}")

### 5.1 Is the difference real?

A paired bootstrap resamples both models over the **same** sequences, so the interval
describes the *difference* rather than the two scores independently. Reading two marginal
confidence intervals and checking whether they overlap is not a test: overlapping intervals
do not imply no difference, and non-overlapping ones are conservative.

**Both AUROC and AP are reported, and they disagree.** That disagreement is the finding, and
it is defensible. AUROC integrates the whole ranking and is dominated by the bulk of easy
negatives. AP weights the top of the ranking, which is where a detector is actually operated:
an analyst reads the highest-scoring sequences, not the median one. A model that leads on AP
while tying on AUROC orders its most confident calls better.

Do not quote one metric without the other.

In [ ]:
DETECTOR = "T-GNN baseline (window max)"
print(f"paired bootstrap, {N_BOOT} resamples over {len(y_test)} test sequences")
print(f"{DETECTOR} minus each baseline\n")
print(f"{'baseline':<32} {'AUROC delta':>28} {'AP delta':>28}")
print("-" * 92)
for name in ("final-state features (6)", "temporal features (7)", "all features (13)"):
    out = []
    for metric in ("auroc", "ap"):
        d, lo, hi = paired_boot(y_test, seq_scores[DETECTOR], seq_scores[name],
                                N_BOOT, metric=metric)
        tag = "sig" if (lo > 0 or hi < 0) else "ns"
        out.append(f"{d:+.4f} [{lo:+.4f},{hi:+.4f}] {tag}")
    print(f"{name:<32} {out[0]:>28} {out[1]:>28}")

print("""
Expected on experiments/run_burst2k_base (seed 42):

  vs final-state features (6)   AUROC +0.3468 [+0.2907,+0.4030] sig   AP +0.4259 [+0.3714,+0.4799] sig
  vs temporal features (7)      AUROC +0.0193 [-0.0297,+0.0719] ns    AP +0.1061 [+0.0454,+0.1708] sig
  vs all features (13)          AUROC +0.0098 [-0.0434,+0.0632] ns    AP +0.0968 [+0.0375,+0.1620] sig
""")

## 6. The other baselines

Three of these are the baselines the original thesis publishes -- Isolation Forest, One-Class
SVM, and a static GNN with the temporal components removed. All three were originally measured
on `llm_generated` and have been re-run here.

**The static GNN is the ablation that matters.** `ap_burst2k` is built so a single snapshot
carries no signal, so a model that reads only the last snapshot should sit near chance while
the full detector reaches 0.81. That isolates the temporal components using the thesis's own
baseline rather than a feature set chosen after the fact.

It trains through the **same `Trainer`** as the detector -- same loss, optimiser, batch size,
augmentation, epoch budget and checkpoint-selection rule -- with the same input features and
the same attention readout. The only difference is that `NodeMemoryModule` and
`TemporalAggregator` are gone.

Two honest caveats to carry into the write-up:

- The static model has **306,243 parameters against the detector's 670,915**. You cannot
  remove the temporal machinery and hold parameter count fixed, so "it is smaller" is an
  available counter-explanation and should be conceded rather than left for an examiner.
- It is run under **both** readouts. With mean pooling only, a loss could be blamed on the
  readout rather than the missing temporal component -- and Sections 2.6 and 3.8 argue that
  mean pooling attenuates exactly this kind of small anomalous subgraph. The result holds
  either way, which is what closes that objection.

In [ ]:
RUN_UNSUPERVISED = False   # True re-runs Isolation Forest and One-Class SVM (~15 min, CPU)
RUN_STATIC_GNN = False     # True retrains both static baselines (~6 h total)

if RUN_UNSUPERVISED:
    subprocess.run([sys.executable, "-m", "src.experiments.unsupervised_baselines",
                    "--data", "data/raw/ap_burst2k"], check=True)

if RUN_STATIC_GNN:
    for tag, readout in (("attn", "attention"), ("mean", "mean")):
        out_dir = f"experiments/run_burst2k_static_{tag}"
        subprocess.run([sys.executable, "-m", "src.experiments.train_static_baseline",
                        "--data", "data/raw/ap_burst2k", "--output", out_dir,
                        "--readout", readout, "--epochs-supervised", str(EPOCHS_SUPERVISED),
                        "--seed", str(SEED)], check=True)
        subprocess.run([sys.executable, "-m", "src.experiments.sequence_eval",
                        "--data", "data/raw/ap_burst2k", "--run-dir", out_dir,
                        "--window-scores", f"{out_dir}/window_scores_test.npz",
                        "--label", f"static GNN ({readout})"], check=True)

unsup_path = DATA_DIR / "unsupervised_baselines.json"
if unsup_path.exists():
    unsup = json.load(open(unsup_path))
    print("RECORD LEVEL -- are injected records distributional outliers?")
    if not unsup["record_level"]:
        # Guard against a half-populated artefact. The CLI has a --skip-record flag, and a
        # run using it overwrites this file with an empty record_level; printing the expected
        # values underneath with nothing measured beside them would read as a match.
        print("  MISSING -- rerun WITHOUT --skip-record:")
        print("    python -m src.experiments.unsupervised_baselines --data data/raw/ap_burst2k")
    for name, r in unsup["record_level"].items():
        print(f"  {name:<20} AUROC {r['auroc']:.4f}   AP {r['ap']:.4f}")
    print("  expected: Isolation Forest 0.5029 / 0.0275, One-Class SVM 0.5531 / 0.0284")
    print("  base rate AP 0.0250 -- both at or near chance, which is the claim that survives")
    print("\nSEQUENCE LEVEL -- fitted on benign training sequences only")
    for name, r in unsup["sequence_level"].items():
        print(f"  {name:<20} AUROC {r['auroc']:.4f}   AP {r['ap']:.4f}   "
              f"CI [{r['ci95'][0]:.4f}, {r['ci95'][1]:.4f}]")
    print("  expected: Isolation Forest 0.6538 / 0.5449, One-Class SVM 0.5719 / 0.5044")
    print("\n  Both beat chance here because the 13-feature table hands them the hand-built")
    print("  temporal summaries. They are unsupervised DETECTORS given engineered FEATURES.")
else:
    print(f"(not found: {unsup_path} -- set RUN_UNSUPERVISED = True)")

In [ ]:
# Load the static-GNN sequence scores and pair them against the detector.
EXPECT_STATIC = {"attention": (0.5602, 0.4684, "+0.2517", "+0.3464"),
                 "mean": (0.5424, 0.4652, "+0.2700", "+0.3503")}

print(f"{'model':<36} {'AUROC':>7} {'AP':>7}   {'paired vs T-GNN (AUROC / AP)':>34}")
print("-" * 92)
for tag, readout in (("attn", "attention"), ("mean", "mean")):
    run_dir = REPO_ROOT / "experiments" / f"run_burst2k_static_{tag}"
    npz = run_dir / "sequence_scores_test.npz"
    if not npz.exists():
        print(f"(not found: {npz} -- set RUN_STATIC_GNN = True)")
        continue
    z = np.load(npz, allow_pickle=True)
    assert (z["y"] == y_test).all(), "label mismatch -- different split?"
    for agg in ("max", "mean", "last"):
        key = f"static_GNN_({readout})_(window_{agg})"
        if key not in z.files:
            continue
        s = z[key]
        seq_scores[key.replace("_", " ")] = s
        deltas = []
        for metric in ("auroc", "ap"):
            d, lo, hi = paired_boot(y_test, seq_scores[DETECTOR], s, N_BOOT, metric=metric)
            deltas.append(f"{d:+.4f}{'*' if (lo > 0 or hi < 0) else ''}")
        print(f"{key.replace('_', ' '):<36} {roc_auc_score(y_test, s):>7.4f} "
              f"{average_precision_score(y_test, s):>7.4f}   "
              f"{deltas[0]:>16} {deltas[1]:>16}")
    e = EXPECT_STATIC[readout]
    print(f"    expected (window max): {e[0]:.4f} / {e[1]:.4f}, "
          f"paired {e[2]} AUROC / {e[3]} AP, both significant")
print("\n* = 95% CI of the paired difference excludes zero")

## 7. Calibrated detection latency

This replaces the original protocol, which used a fixed threshold of 0.5 with **no false-alarm
rate stated**. A latency quoted at an unknown FPR is uninterpretable: any detector can be made
to look fast by lowering its threshold.

Four things are fixed here:

1. The threshold is **calibrated on benign validation windows** to a target FPR, and the
   realised rate on benign *test* windows is reported beside it.
2. Both detectors are treated identically -- fit on train, calibrate on validation, evaluate
   on test.
3. Sequences that are **never detected are kept as censored**, not dropped. Dropping them
   flatters whichever detector misses more.
4. Alarms raised **before the attack exists** count as false alarms, not early detections.

The head-to-head is then repeated with thresholds set so the realised FPR is identical by
construction, because validation calibration transfers differently for each model.

In [ ]:
RUN_LATENCY = False   # True recomputes (~10 min, needs the checkpoint)

if RUN_LATENCY:
    subprocess.run([sys.executable, "-m", "src.experiments.detection_latency",
                    "--data", "data/raw/ap_burst2k",
                    "--run-dir", "experiments/run_burst2k_base",
                    "--target-fpr", "0.05", "0.10"], check=True)

lat_path = EXP_DIR / "latency_matched_fpr.json"
if lat_path.exists():
    print(json.dumps(json.load(open(lat_path)), indent=2)[:1200])
else:
    print(f"(not found: {lat_path} -- set RUN_LATENCY = True)")

print("""
Recorded in the thesis, at a realised benign-window FPR fixed at 5% for BOTH detectors:

  detector        detected        median latency   caught within k snapshots of onset
                                                   k=0     k=1     k=2     k=5     k=10
  9 features (LR) 56/200 (28.0%)       1.0         0.010   0.150   0.210   0.275   0.280
  T-GNN          142/200 (71.0%)       1.0         0.120   0.385   0.480   0.665   0.710

  paired cluster bootstrap over the 200 poisoned sequences, T-GNN minus LR:
    k=0  +0.110 [+0.070,+0.155]      k=5  +0.389 [+0.320,+0.460]
    k=1  +0.235 [+0.175,+0.300]      k=10 +0.429 [+0.360,+0.495]
    k=2  +0.269 [+0.205,+0.335]
  significant at every horizon.

A reading to include: under validation calibration the LR's median latency of 1.5 looks
competitive against the T-GNN's 2.0. That is survivorship -- the LR detects only the easiest
30% and takes a median over that gentler subset. At matched FPR both medians are 1.0 and the
illusion disappears.
""")

## 8. The table for the thesis

Everything assembled in one place, sorted by AUROC. This is the comparison to reproduce in
Chapter 4.

In [ ]:
rows = []
for name, s in seq_scores.items():
    if name.startswith("T-GNN"):
        kind = "proposed"
    elif name.startswith("static GNN"):
        kind = "static GNN (ablation)"
    else:
        kind = "logistic regression"
    rows.append((name, kind, roc_auc_score(y_test, s), average_precision_score(y_test, s)))

if unsup_path.exists():
    for name, r in json.load(open(unsup_path))["sequence_level"].items():
        rows.append((name, "unsupervised", r["auroc"], r["ap"]))

rows.sort(key=lambda r: -r[2])
print(f"{'model':<38} {'kind':<24} {'AUROC':>7} {'AP':>7}")
print("-" * 80)
for name, kind, auroc, apr in rows:
    mark = " <--" if name == DETECTOR else ""
    print(f"{name:<38} {kind:<24} {auroc:>7.4f} {apr:>7.4f}{mark}")

print("""
HOW TO READ THIS

The only baselines that match the detector on AUROC are the temporal and 13-feature logistic
regressions -- and both are ORACLES, hand-built from knowledge of how the attack is generated.
Every baseline not given that knowledge is beaten significantly on both metrics: the static
GNN by +0.25 AUROC, Isolation Forest by +0.16, One-Class SVM by +0.24.

That is the claim to make in the thesis. The detector matches a baseline that was told the
answer, and beats everything that was not.

Nothing here is at 1.0000, and nothing should be. The previous benchmark produced a perfect
score for every model, which is precisely how it was found to be defective: a spread across
models is evidence, and a ceiling is the absence of it.
""")